In [ ]:
import torch
import torch.nn as nn
from torch import Tensor

def block_attn_res(blocks: list[Tensor], partial_block: Tensor, proj: nn.Linear, norm: nn.Module) -> Tensor:
    """
    Rewritten Inter-block attention without einsum.
    """
    # 1. Stack all representations
    # V shape: [N+1, B, T, D] (where N is the number of completed blocks)
    V = torch.stack(blocks + [partial_block]) 
    
    # 2. Normalize
    # K shape: [N+1, B, T, D]
    K = norm(V)
    
    # 3. Compute attention logits
    # Assuming proj is an nn.Linear(D, 1), proj(K) yields [N+1, B, T, 1].
    # Squeezing the last dimension gives [N+1, B, T].
    # (This matches the original einsum dot product across 'd')
    logits = proj(K).squeeze(dim=-1) 
    
    # 4. Compute attention weights
    # Softmax across the block dimension (dim=0)
    # attn_weights shape: [N+1, B, T]
    attn_weights = logits.softmax(dim=0)
    
    # 5. Apply weights and sum
    # Unsqueeze to [N+1, B, T, 1] so it broadcasts with V [N+1, B, T, D]
    # Multiply element-wise, then sum across the block dimension (dim=0)
    # h shape: [B, T, D]
    h = (attn_weights.unsqueeze(-1) * V).sum(dim=0)
    
    return h

# The forward pass doesn't use einsum, but here it is for completeness
def forward(self, blocks: list[Tensor], hidden_states: Tensor) -> tuple[list[Tensor], Tensor]:
    partial_block = hidden_states
    
    # 1. Block Attention before Self-Attention
    h = block_attn_res(blocks, partial_block, self.attn_res_proj, self.attn_res_norm)
    
    # 2. Check Block Boundary
    if self.layer_number % (self.block_size // 2) == 0:
        blocks.append(partial_block)
        partial_block = None
        
    # 3. Self-Attention Layer
    attn_out = self.attn(self.attn_norm(h))
    partial_block = partial_block + attn_out if partial_block is not None else attn_out
    
    # 4. Block Attention before MLP
    h = block_attn_res(blocks, partial_block, self.mlp_res_proj, self.mlp_res_norm)
    
    # 5. MLP Layer
    mlp_out = self.mlp(self.mlp_norm(h))
    partial_block = partial_block + mlp_out
    
    return blocks, partial_block